# GDELT 라이브러리 End-to-End 테스트

**목적**: `gdelt` 라이브러리로 실제 GKG 데이터를 수집해,  
`ML_DataPrep_Example.ipynb`의 필터링·집계 파이프라인이 올바르게 동작하는지 검증합니다.

**커널**: Finance Project (.venv) — `gdelt`는 `finance_project/.venv`에 설치됨

**테스트 전략**
| 모드 | 행 수/일 | 속도 | 용도 |
|------|---------|------|------|
| `coverage=False` | ~1,600행 | 빠름 (초 단위) | 구조 확인, 컬럼 파악 |
| `coverage=True`  | ~126,000행 | 느림 (분 단위) | 커버리지 검증, 실제 테스트 |

→ **Step 1~3**: `coverage=False`로 빠르게 구조 파악  
→ **Step 4~6**: `coverage=True` 1일치로 실제 커버리지 측정

---
### 실 데이터 테스트 결과 요약 (2025-04-14 기준)
| 항목 | 결과 |
|------|------|
| 수집 행 수 | 126,375행 |
| 커버 종목 | 14 / 30 (47%) |
| 최다 기사 | GOOGL 1,872건, QQQ 1,295건, MSFT 615건 |
| 채권 ETF | 0건 → 테마 키워드 조정 필요 |
| GLD | 0건 → ECON_GOLD 테마 부재, raw GOLD 1,292건 있으나 오매핑 위험 |

## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import platform
import warnings
import gdelt
warnings.filterwarnings('ignore')

# ── 한글 폰트 ─────────────────────────────────────────────────
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    import koreanize_matplotlib
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

TEST_DATE = '2025 Apr 14'

TICKERS = [
    'SPY','QQQ','IWM','EFA','EEM',
    'TLT','AGG','SHY','TIP',
    'GLD','DBC',
    'XLK','XLF','XLE','XLV','VOX','XLY','XLP','XLI','XLU','XLRE','XLB',
    'AAPL','MSFT','AMZN','GOOGL','JPM','JNJ','PG','XOM'
]

gd = gdelt.gdelt(version=2)
print(f'환경 설정 완료  |  종목 수: {len(TICKERS)}개')

## Step 1. GKG 수집 및 컬럼 구조 파악

`coverage=False`: 하루 마지막 15분 파일 1개 (~1,600행) — 빠른 구조 확인용

In [ ]:
print(f'수집 중: {TEST_DATE} (coverage=False)...')
gkg_sample = gd.Search([TEST_DATE], table='gkg', coverage=False)
print(f'수집 완료: {gkg_sample.shape[0]:,}행 × {gkg_sample.shape[1]}열')

print('\n전체 컬럼 목록:')
for i, col in enumerate(gkg_sample.columns):
    print(f'  [{i:2d}] {col}')

In [ ]:
# ── 핵심 컬럼 탐색 (gdelt 라이브러리 버전에 따라 컬럼명 상이할 수 있음) ──
def find_col(df: pd.DataFrame, candidates: list) -> str:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    raise KeyError(f'후보 {candidates} 중 없음. 실제 컬럼: {list(df.columns)}')

COL_DATE   = find_col(gkg_sample, ['DATE', 'date'])
COL_THEMES = find_col(gkg_sample, ['Themes', 'THEMES', 'V2Themes'])
COL_ORGS   = find_col(gkg_sample, ['Organizations', 'ORGANIZATIONS', 'AllNames'])
COL_TONE   = find_col(gkg_sample, ['V2Tone', 'Tone', 'tone'])

print('=== 핵심 컬럼 매핑 결과 (실제 확인값) ===')
print(f'  날짜 컬럼: "{COL_DATE}"')
print(f'  테마 컬럼: "{COL_THEMES}"')
print(f'  기관 컬럼: "{COL_ORGS}"')
print(f'  감성 컬럼: "{COL_TONE}"')

print(f'\n{COL_TONE} 샘플:')
print(gkg_sample[COL_TONE].dropna().head(2).tolist())
print(f'\n{COL_ORGS} 샘플:')
print(gkg_sample[COL_ORGS].dropna().head(2).tolist())
print(f'\n{COL_THEMES} 샘플:')
print(gkg_sample[COL_THEMES].dropna().head(2).tolist())

## Step 2. 종목별 매핑 테이블 + 필터링 함수 정의

### 매핑 타입 3가지
| 타입 | 대상 | 방식 |
|------|------|------|
| `org` | 개별 대형주 | Organizations에서 기업명 직접 검색 |
| `org_themed` | AMZN (지리명 오매핑 위험) | 기업명 AND 경제/기술 테마 |
| `keyword` | ETF류 | Themes 태그 AND 키워드 조합 |

In [ ]:
TICKER_TO_GDELT = {

    # ── 개별 대형주: Organizations 기업명 직접 매핑 ────────────
    'AAPL': {'type':'org', 'names':['APPLE INC', 'APPLE COMPUTER']},
    'MSFT': {'type':'org', 'names':['MICROSOFT CORP', 'MICROSOFT CORPORATION', 'MICROSOFT']},

    # AMZN: org_themed — 'AMAZON' 단독 키워드는 지리명(Amazon River)과 혼용 위험
    # → 경제/기술/리테일 테마 필터와 AND 조합으로 노이즈 차단
    'AMZN': {'type':'org_themed',
             'names':  ['AMAZON.COM', 'AMAZON INC', 'AMAZON WEB SERVICES', 'AWS'],
             'themes': ['ECON_STOCKMARKET', 'TECHNOLOGY', 'ECON_RETAIL']},

    'GOOGL':{'type':'org', 'names':['ALPHABET INC', 'GOOGLE', 'ALPHABET']},
    'JPM':  {'type':'org', 'names':['JPMORGAN CHASE', 'JP MORGAN', 'JPMORGAN']},
    'JNJ':  {'type':'org', 'names':['JOHNSON & JOHNSON', 'JOHNSON AND JOHNSON']},
    'PG':   {'type':'org', 'names':['PROCTER & GAMBLE', 'PROCTER AND GAMBLE']},
    'XOM':  {'type':'org', 'names':['EXXON MOBIL', 'EXXONMOBIL', 'EXXON']},

    # ── 광범위 시장 ETF ──────────────────────────────────────────
    'SPY':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['S&P 500', 'S&P500', 'STOCK MARKET']},
    'QQQ':  {'type':'keyword', 'themes':['ECON_STOCKMARKET', 'TECHNOLOGY'],
             'keywords':['NASDAQ', 'TECH STOCKS', 'TECHNOLOGY SECTOR']},
    'IWM':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['RUSSELL 2000', 'SMALL CAP', 'SMALL-CAP']},
    'EFA':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['MSCI EAFE', 'DEVELOPED MARKET', 'INTERNATIONAL EQUITY']},
    'EEM':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['MSCI EMERGING', 'EMERGING MARKET', 'EM EQUITY']},

    # ── 섹터 ETF ─────────────────────────────────────────────────
    'XLK':  {'type':'keyword', 'themes':['TECHNOLOGY'],
             'keywords':['TECHNOLOGY SECTOR', 'TECH SECTOR', 'INFORMATION TECHNOLOGY']},
    'XLF':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['FINANCIAL SECTOR', 'BANKING SECTOR', 'FINANCIAL STOCKS']},
    'XLE':  {'type':'keyword', 'themes':['ENV_OIL', 'ENERGY'],
             'keywords':['ENERGY SECTOR', 'OIL STOCKS', 'ENERGY STOCKS']},
    'XLV':  {'type':'keyword', 'themes':['HEALTH'],
             'keywords':['HEALTH CARE SECTOR', 'HEALTHCARE STOCKS', 'PHARMA SECTOR']},
    'VOX':  {'type':'keyword', 'themes':['TECHNOLOGY'],
             'keywords':['COMMUNICATION SERVICES', 'TELECOM SECTOR']},
    'XLY':  {'type':'keyword', 'themes':['ECON_RETAIL'],
             'keywords':['CONSUMER DISCRETIONARY', 'CONSUMER SPENDING', 'RETAIL SECTOR']},
    'XLP':  {'type':'keyword', 'themes':['ECON_RETAIL'],
             'keywords':['CONSUMER STAPLES', 'DEFENSIVE STOCKS', 'STAPLES SECTOR']},
    'XLI':  {'type':'keyword', 'themes':['ECON_TRADE'],
             'keywords':['INDUSTRIAL SECTOR', 'INDUSTRIALS', 'MANUFACTURING SECTOR']},
    'XLU':  {'type':'keyword', 'themes':['ENERGY'],
             'keywords':['UTILITIES SECTOR', 'ELECTRIC UTILITIES', 'UTILITY STOCKS']},
    'XLRE': {'type':'keyword', 'themes':['ECON_REALESTATE'],
             'keywords':['REAL ESTATE SECTOR', 'REIT SECTOR', 'PROPERTY SECTOR']},
    'XLB':  {'type':'keyword', 'themes':['COMMODITY'],
             'keywords':['MATERIALS SECTOR', 'BASIC MATERIALS', 'CHEMICALS SECTOR']},

    # ── 채권 ETF: 금리 키워드 (커버리지 확인 필요) ───────────────
    # 실 테스트 결과 0건 → Themes 컬럼에 ECON_INTEREST_RATES 태그 부재 가능성
    # Step 6에서 실제 Themes 태그 분포를 확인 후 키워드 조정 예정
    'TLT':  {'type':'keyword', 'themes':['ECON_INTEREST_RATES'],
             'keywords':['20-YEAR TREASURY', '30-YEAR BOND', 'LONG-TERM TREASURY']},
    'AGG':  {'type':'keyword', 'themes':['ECON_INTEREST_RATES'],
             'keywords':['INVESTMENT GRADE', 'BOND MARKET', 'AGGREGATE BOND']},
    'SHY':  {'type':'keyword', 'themes':['ECON_INTEREST_RATES'],
             'keywords':['2-YEAR TREASURY', 'SHORT-TERM TREASURY', 'FED FUNDS RATE']},
    'TIP':  {'type':'keyword', 'themes':['ECON_INTEREST_RATES', 'ECON_INFLATION'],
             'keywords':['TIPS', 'INFLATION PROTECTED', 'REAL YIELD']},

    # ── 대안 ETF (커버리지 확인 필요) ────────────────────────────
    # GLD: ECON_GOLD 태그 부재 가능성 → Step 6에서 GOLD 관련 Themes 태그 확인 예정
    'GLD':  {'type':'keyword', 'themes':['ECON_GOLD', 'COMMODITY'],
             'keywords':['GOLD PRICE', 'PRECIOUS METALS', 'GOLD MARKET']},
    'DBC':  {'type':'keyword', 'themes':['COMMODITY'],
             'keywords':['COMMODITY INDEX', 'RAW MATERIALS', 'COMMODITY MARKET']},
}

print(f'매핑 정의 종목: {len(TICKER_TO_GDELT)}개 / 전체 {len(TICKERS)}개')

In [ ]:
def extract_gdelt_for_ticker(gkg: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    GKG DataFrame에서 특정 티커에 해당하는 행을 추출합니다.

    타입별 매핑 전략:
      org        : Organizations 컬럼에서 기업명 직접 검색
      org_themed : 기업명 AND 테마 조합 (AMZN 지리명 오매핑 차단)
      keyword    : Themes 태그 AND 키워드 조합 (ETF류)
    """
    if ticker not in TICKER_TO_GDELT:
        return pd.DataFrame()

    config = TICKER_TO_GDELT[ticker]

    if config['type'] == 'org':
        pattern = '|'.join(config['names'])
        mask = gkg[COL_ORGS].str.contains(pattern, na=False, case=False)

    elif config['type'] == 'org_themed':
        # 기업명 매칭 AND 테마 필터 (AMZN: Amazon River 등 오매핑 방지)
        name_pattern = '|'.join(config['names'])
        mask_name = gkg[COL_ORGS].str.contains(name_pattern, na=False, case=False)
        theme_pattern = '|'.join(config['themes'])
        mask_theme = gkg[COL_THEMES].str.contains(theme_pattern, na=False, case=False)
        mask = mask_name & mask_theme

    elif config['type'] == 'keyword':
        theme_pattern = '|'.join(config['themes'])
        mask_theme = gkg[COL_THEMES].str.contains(theme_pattern, na=False, case=False)
        kw_pattern = '|'.join(config['keywords'])
        mask_kw = (
            gkg[COL_ORGS].str.contains(kw_pattern, na=False, case=False) |
            gkg[COL_THEMES].str.contains(kw_pattern, na=False, case=False)
        )
        mask = mask_theme & mask_kw
    else:
        return pd.DataFrame()

    matched = gkg[mask].copy()
    matched['ticker'] = ticker
    return matched


def aggregate_gdelt_by_stock(gkg: pd.DataFrame) -> pd.DataFrame:
    """
    GKG 전체를 (date, ticker) 기준으로 집계합니다.

    Returns
    -------
    컬럼: date, ticker,
          gdelt_tone_avg, gdelt_tone_neg, gdelt_tone_polar,
          gdelt_event_cnt, gdelt_tone_std
    """
    tone_split = gkg[COL_TONE].str.split(',', expand=True)
    gkg = gkg.copy()
    gkg['_tone_avg']   = pd.to_numeric(tone_split[0], errors='coerce')
    gkg['_tone_neg']   = pd.to_numeric(tone_split[2], errors='coerce')
    gkg['_tone_polar'] = pd.to_numeric(tone_split[3], errors='coerce')
    gkg['_date'] = pd.to_datetime(
        gkg[COL_DATE].astype(str).str[:8], format='%Y%m%d', errors='coerce'
    )

    all_rows = []
    for ticker in TICKERS:
        matched = extract_gdelt_for_ticker(gkg, ticker)
        if not matched.empty:
            all_rows.append(matched)

    if not all_rows:
        print('경고: 매핑된 데이터 없음')
        return pd.DataFrame()

    gdelt_mapped = pd.concat(all_rows, ignore_index=True)

    return (
        gdelt_mapped
        .groupby(['_date', 'ticker'])
        .agg(
            gdelt_tone_avg   = ('_tone_avg',   'mean'),
            gdelt_tone_neg   = ('_tone_neg',   'mean'),
            gdelt_tone_polar = ('_tone_polar', 'mean'),
            gdelt_event_cnt  = ('_tone_avg',   'count'),
            gdelt_tone_std   = ('_tone_avg',   'std'),
        )
        .reset_index()
        .rename(columns={'_date': 'date'})
    )


print('함수 정의 완료')

## Step 3. 빠른 구조 테스트 (coverage=False)

In [ ]:
result_fast = aggregate_gdelt_by_stock(gkg_sample)

print(f'집계 결과: {result_fast.shape}')
print(f'커버 종목: {sorted(result_fast["ticker"].unique().tolist())}')
result_fast.sort_values('gdelt_event_cnt', ascending=False)

## Step 4. 1일치 전체 커버리지 테스트 (coverage=True)

> ⚠️ **Jupyter에서는 `if __name__` 불필요** — 노트북 커널은 단일 프로세스로 실행됩니다.  
> ⚠️ 약 2~4분 소요됩니다.

In [ ]:
print(f'수집 중: {TEST_DATE} (coverage=True, 전체 파일)...')
gkg_full = gd.Search([TEST_DATE], table='gkg', coverage=True)
print(f'수집 완료: {gkg_full.shape[0]:,}행 × {gkg_full.shape[1]}열')

In [ ]:
print('종목별 필터링 및 집계 실행 중...')
result_full = aggregate_gdelt_by_stock(gkg_full)

print(f'\n집계 결과: {result_full.shape}')
result_full.sort_values('gdelt_event_cnt', ascending=False).head(20)

## Step 5. 커버리지 시각화

In [ ]:
cnt_full = result_full.groupby('ticker')['gdelt_event_cnt'].sum().reindex(TICKERS, fill_value=0)

type_color = {}
for t in TICKERS:
    if t in ['AAPL','MSFT','AMZN','GOOGL','JPM','JNJ','PG','XOM']:
        type_color[t] = '#2196F3'
    elif t in ['TLT','AGG','SHY','TIP']:
        type_color[t] = '#FF9800'
    elif t in ['GLD','DBC']:
        type_color[t] = '#9C27B0'
    else:
        type_color[t] = '#4CAF50'

fig, ax = plt.subplots(figsize=(14, 5))
colors = [type_color.get(t, 'gray') for t in cnt_full.index]
cnt_full.plot(kind='bar', ax=ax, color=colors, alpha=0.85)
ax.set_title(f'종목별 GDELT 기사 건수 (coverage=True, {TEST_DATE})', fontsize=12)
ax.set_ylabel('기사 건수')
ax.tick_params(axis='x', rotation=45)
legend_items = [
    Patch(color='#2196F3', label='개별 대형주'),
    Patch(color='#4CAF50', label='시장/섹터 ETF'),
    Patch(color='#FF9800', label='채권 ETF'),
    Patch(color='#9C27B0', label='대안 ETF'),
]
ax.legend(handles=legend_items, fontsize=9)
plt.tight_layout()
plt.show()

actual_cnt = (cnt_full > 0).sum()
print(f'커버 종목: {actual_cnt}/{len(TICKERS)}개 ({actual_cnt/len(TICKERS)*100:.0f}%)')
print('\n미커버 종목:')
print([t for t in TICKERS if cnt_full[t] == 0])

In [ ]:
# ── 종목별 평균 감성 ──────────────────────────────────────────
if not result_full.empty:
    tone_by_ticker = result_full.groupby('ticker')['gdelt_tone_avg'].mean().reindex(TICKERS)
    colors_tone = ['#d32f2f' if (v < -2) else ('#1976d2' if (v > 0) else '#757575')
                   for v in tone_by_ticker.fillna(0)]

    fig, ax = plt.subplots(figsize=(12, 4))
    tone_by_ticker.plot(kind='bar', ax=ax, color=colors_tone, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'종목별 평균 감성 (tone_avg, {TEST_DATE})', fontsize=12)
    ax.set_ylabel('평균 Tone  (음수=부정, 양수=긍정)')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## Step 6. 노이즈 점검 및 커버리지 미달 종목 원인 분석

### 6-A. 채권 ETF / GLD 미커버 원인 조사

실 테스트 결과 TLT, AGG, GLD 등이 0건인 원인을 파악합니다.  
GKG `Themes` 컬럼에 실제로 어떤 태그들이 사용되는지 확인합니다.

In [ ]:
# ── GKG Themes 컬럼 전체 분포 분석 ────────────────────────────────
# coverage=True 전체 데이터의 Themes 태그 inventory를 먼저 파악해야
# 올바른 테마 코드를 특정할 수 있습니다.

all_themes_series = (
    gkg_full[COL_THEMES]
    .dropna()
    .str.split(';')
    .explode()
    .str.strip()
    .loc[lambda s: s != '']
)

total_tokens = len(all_themes_series)
unique_count = all_themes_series.nunique()
print(f'Themes 토큰 총계: {total_tokens:,}개  |  고유 테마 수: {unique_count:,}개')

# ── 테마 접두사(taxonomy 시스템) 분포 ────────────────────────────
# GDELT GKG v2 는 복수의 taxonomy를 혼합 사용:
#   ECON_  : GDELT 경제 테마
#   EPU_   : Economic Policy Uncertainty 분류
#   ENV_   : 환경/에너지/자원
#   TAX_   : 기능/행위 분류
#   WB_    : World Bank Development Indicators
#   CRISISLEX_ : 위기/재난 분류
import re
prefix_series = all_themes_series.str.extract(r'^([A-Z]+(?:_[A-Z0-9]+)?)', expand=False)
prefix_dist = prefix_series.dropna().value_counts()
print('\n[접두사별 분포 TOP 40 — taxonomy 시스템 파악]')
print(prefix_dist.head(40).to_string())

# ── ECON_ 계열 태그 ───────────────────────────────────────────────
econ_themes = all_themes_series[all_themes_series.str.startswith('ECON')].value_counts()
print(f'\n[ECON_ 계열 | {len(econ_themes)}종 — 동작 확인된 태그: ECON_STOCKMARKET]')
print(econ_themes.head(30).to_string() if not econ_themes.empty else '  없음')

In [ ]:
# ── EPU_ 계열 (Economic Policy Uncertainty) ──────────────────────
# 금리/채권/재정/통화정책 기사에 자주 등장하는 태그 체계
# 실 테스트에서 AMZN/AAPL 기사에 EPU_ECONOMY가 등장 → 동작 확인됨
epu_themes = all_themes_series[all_themes_series.str.startswith('EPU')].value_counts()
print(f'[EPU_ 계열 | {len(epu_themes)}종]')
print(epu_themes.head(25).to_string() if not epu_themes.empty else '  EPU 태그 없음')

# ── ENV_ 계열 (환경/에너지/자원) ─────────────────────────────────
# ENV_OIL, ENERGY는 XLE에서 동작 확인됨
env_themes = all_themes_series[all_themes_series.str.startswith('ENV')].value_counts()
print(f'\n[ENV_ 계열 | {len(env_themes)}종]')
print(env_themes.head(20).to_string() if not env_themes.empty else '  ENV 태그 없음')

# ── 금리/채권/통화정책 관련 테마 키워드 검색 ──────────────────────
rate_kw = 'INTEREST|RATE|BOND|TREASURY|YIELD|MONETARY|DEBT|FEDERAL'
rate_themes = all_themes_series[
    all_themes_series.str.contains(rate_kw, case=False)
].value_counts()
print(f'\n[금리/채권 관련 테마]')
print(rate_themes.head(20).to_string() if not rate_themes.empty else '  해당 태그 없음 ← ECON_INTEREST_RATES 실제 미존재 가능성 높음')

# ── 금/원자재 관련 테마 키워드 검색 ──────────────────────────────
gold_kw = 'GOLD|PRECIOUS|COMMODIT|MINING|METAL|SILVER'
gold_themes = all_themes_series[
    all_themes_series.str.contains(gold_kw, case=False)
].value_counts()
print(f'\n[금/원자재 관련 테마]')
print(gold_themes.head(20).to_string() if not gold_themes.empty else '  해당 태그 없음 ← ECON_GOLD 실제 미존재 가능성 높음')

# ── 헬스케어 관련 테마 키워드 검색 ──────────────────────────────
health_kw = 'HEALTH|PHARMA|MEDICAL|BIOTECH|WELLNESS'
health_themes = all_themes_series[
    all_themes_series.str.contains(health_kw, case=False)
].value_counts()
print(f'\n[헬스케어 관련 테마]')
print(health_themes.head(15).to_string() if not health_themes.empty else '  해당 태그 없음 ← HEALTH 테마 실제 미존재 가능성 높음')

In [ ]:
# ── Organizations 컬럼에서 'GOLD' 포함 행 원본 확인 ────────────
# ECON_GOLD 태그 없을 경우 직접 Organizations 검색으로 대체 가능 여부 확인

gold_raw = gkg_full[
    gkg_full[COL_ORGS].str.contains('GOLD', na=False, case=False)
]
print(f"'GOLD' 포함 Organizations 행: {len(gold_raw)}건")
print('\n샘플 (기관명 원본):')
for v in gold_raw[COL_ORGS].dropna().head(8).values:
    print(f'  {str(v)[:100]}')

# → GOLDMAN SACHS, GOLDEN STATE 등 오매핑 케이스 확인

In [ ]:
# ── 채권/금리 관련 Organizations 원본 확인 ────────────────────
treasury_raw = gkg_full[
    gkg_full[COL_ORGS].str.contains('TREASURY|FEDERAL RESERVE|FED ', na=False, case=False)
]
print(f"금리/채권 관련 Organizations 행: {len(treasury_raw)}건")
print('\n샘플:')
for v in treasury_raw[COL_ORGS].dropna().head(5).values:
    print(f'  {str(v)[:100]}')

### 6-B. AAPL 노이즈 점검 + JNJ/PG 미커버 원인 조사

- **AAPL (254건)**: "apple app", "apple store" 등 비기업 매핑 비율 확인
- **JNJ/PG (0건)**: `org` 타입인데도 0건 → GDELT Organizations에서 실제 기업명 형태 직접 확인
  - 가설: `&` 기호가 GDELT 인코딩에서 누락될 수 있음

In [ ]:
# ── AAPL 노이즈 점검 ─────────────────────────────────────────────
aapl_rows = extract_gdelt_for_ticker(gkg_full, 'AAPL')
print(f'AAPL 매핑 건수: {len(aapl_rows)}건')

# Organizations 컬럼에서 'apple' 포함 항목 빈도 → 오매핑 비율 파악
apple_orgs = (
    aapl_rows[COL_ORGS]
    .dropna()
    .str.lower()
    .str.split(';')
    .explode()
    .str.strip()
)
apple_terms = apple_orgs[apple_orgs.str.contains('apple', na=False)].value_counts()
print('\n[AAPL] Organizations 내 "apple" 포함 항목 (상위 20):')
print(apple_terms.head(20).to_string())

# 'apple inc' vs 나머지(app, store 등) 비율
true_hits = apple_terms[apple_terms.index.str.contains('apple inc|apple computer', na=False)].sum()
noise_hits = apple_terms.sum() - true_hits
print(f'\n  진짜 Apple Inc 관련: {true_hits}건')
print(f'  노이즈 (app/store 등): {noise_hits}건')
print(f'  노이즈 비율: {noise_hits / apple_terms.sum() * 100:.1f}%' if apple_terms.sum() > 0 else '')

# ── JNJ/PG Organizations 실제 형태 직접 확인 ─────────────────────
print('\n' + '='*60)
for ticker, search_kw in [('JNJ', 'JOHNSON'), ('PG', 'PROCTER')]:
    raw = gkg_full[gkg_full[COL_ORGS].str.contains(search_kw, na=False, case=False)]
    print(f"\n[{ticker}] '{search_kw}' raw 검색: {len(raw)}건")
    all_orgs_kw = (
        raw[COL_ORGS].dropna()
        .str.split(';')
        .explode()
        .str.strip()
    )
    kw_orgs = all_orgs_kw[
        all_orgs_kw.str.contains(search_kw, na=False, case=False)
    ].value_counts()
    print(f'  실제 기관명 형태 상위 10 (v2 매핑에 추가할 키워드 후보):')
    print(kw_orgs.head(10).to_string() if not kw_orgs.empty else '  해당 기업명 없음 — 당일 기사 부재 가능성')

### 6-C. 개선된 매핑 테이블 v2 + 커버리지 재검증

Step 6-A/B 분석 결과를 반영한 핵심 수정 사항:

| 대상 종목 | 기존 테마 | 수정 테마 | 수정 이유 |
|-----------|----------|----------|----------|
| TLT/AGG/SHY/TIP | `ECON_INTEREST_RATES` | **`EPU_ECONOMY`** | ECON_INTEREST_RATES 미존재, EPU_ECONOMY 동작 확인 |
| GLD | `ECON_GOLD` + `COMMODITY` | **`ECON_STOCKMARKET`** | 두 테마 모두 미동작 |
| DBC | `COMMODITY` | **`ECON_STOCKMARKET`** | COMMODITY 테마 미동작 |
| XLV | `HEALTH` | **`ECON_STOCKMARKET`** | HEALTH 테마 미동작 |
| XLY/XLP | `ECON_RETAIL` | **`EPU_ECONOMY`** | ECON_RETAIL 미동작 |
| XLI | `ECON_TRADE` | **`ECON_STOCKMARKET`** | ECON_TRADE 미동작 |
| XLRE | `ECON_REALESTATE` | **`ECON_STOCKMARKET`** | ECON_REALESTATE 미동작 |
| XLB | `COMMODITY` | **`ECON_STOCKMARKET`** | COMMODITY 미동작 |
| JNJ/PG | `JOHNSON & JOHNSON` 등 | `JOHNSON JOHNSON` 등 **추가** | `&` 인코딩 누락 대비 |

> ⚠️ **EPU_ECONOMY**는 넓은 경제정책 태그입니다. 키워드 필터로 충분히 좁혀야 노이즈를 제어할 수 있습니다.

In [ ]:
# ── TICKER_TO_GDELT v2: 개선된 매핑 테이블 ────────────────────────
# Step 6 분석 결과 반영:
#   채권 ETF: ECON_INTEREST_RATES(미존재) → EPU_ECONOMY(동작 확인)
#   대안/섹터 ETF: 미동작 테마 → ECON_STOCKMARKET 또는 EPU_ECONOMY
#   JNJ/PG: & 기호 인코딩 문제 대비 키워드 추가

TICKER_TO_GDELT_V2 = {

    # ── 개별 대형주: Organizations 기업명 직접 매핑 ──────────────
    'AAPL': {'type':'org',
             'names':['APPLE INC', 'APPLE COMPUTER']},
    'MSFT': {'type':'org',
             'names':['MICROSOFT CORP', 'MICROSOFT CORPORATION', 'MICROSOFT']},
    # AMZN: org_themed — 지리명(Amazon River) 오매핑 차단
    # ECON_RETAIL 미동작 확인 → EPU_ECONOMY로 교체
    'AMZN': {'type':'org_themed',
             'names':  ['AMAZON.COM', 'AMAZON INC', 'AMAZON WEB SERVICES', 'AWS'],
             'themes': ['ECON_STOCKMARKET', 'TECHNOLOGY', 'EPU_ECONOMY']},
    'GOOGL':{'type':'org',
             'names':['ALPHABET INC', 'GOOGLE', 'ALPHABET']},
    'JPM':  {'type':'org',
             'names':['JPMORGAN CHASE', 'JP MORGAN', 'JPMORGAN']},
    # JNJ/PG: '&' 기호가 GDELT에서 누락될 수 있어 & 없는 버전 추가
    'JNJ':  {'type':'org',
             'names':['JOHNSON & JOHNSON', 'JOHNSON AND JOHNSON', 'JOHNSON JOHNSON']},
    'PG':   {'type':'org',
             'names':['PROCTER & GAMBLE', 'PROCTER AND GAMBLE', 'PROCTER GAMBLE']},
    'XOM':  {'type':'org',
             'names':['EXXON MOBIL', 'EXXONMOBIL', 'EXXON']},

    # ── 광범위 시장 ETF ────────────────────────────────────────────
    'SPY':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['S&P 500', 'S&P500', 'STOCK MARKET', 'EQUITIES']},
    'QQQ':  {'type':'keyword', 'themes':['ECON_STOCKMARKET', 'TECHNOLOGY'],
             'keywords':['NASDAQ', 'TECH STOCKS', 'TECHNOLOGY SECTOR']},
    'IWM':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['RUSSELL 2000', 'SMALL CAP', 'SMALL-CAP']},
    'EFA':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['MSCI EAFE', 'DEVELOPED MARKET', 'INTERNATIONAL EQUITY']},
    'EEM':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['MSCI EMERGING', 'EMERGING MARKET', 'EM EQUITY']},

    # ── 섹터 ETF ──────────────────────────────────────────────────
    'XLK':  {'type':'keyword', 'themes':['TECHNOLOGY'],
             'keywords':['TECHNOLOGY SECTOR', 'TECH SECTOR', 'INFORMATION TECHNOLOGY']},
    'XLF':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['FINANCIAL SECTOR', 'BANKING SECTOR', 'FINANCIAL STOCKS']},
    'XLE':  {'type':'keyword', 'themes':['ENV_OIL', 'ENERGY'],
             'keywords':['ENERGY SECTOR', 'OIL STOCKS', 'ENERGY STOCKS']},
    # XLV: HEALTH(미동작) → ECON_STOCKMARKET + 헬스케어 전문 키워드
    'XLV':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['HEALTH CARE', 'PHARMACEUTICAL', 'BIOTECH', 'MEDICAL DEVICE']},
    # VOX: 키워드 단순화 (COMMUNICATION SERVICES → COMMUNICATION 등)
    'VOX':  {'type':'keyword', 'themes':['TECHNOLOGY'],
             'keywords':['COMMUNICATION', 'TELECOM', 'MEDIA SECTOR', 'BROADCASTING']},
    # XLY/XLP: ECON_RETAIL(미동작) → EPU_ECONOMY (소비 관련 정책 기사 포착)
    'XLY':  {'type':'keyword', 'themes':['EPU_ECONOMY'],
             'keywords':['CONSUMER DISCRETIONARY', 'CONSUMER SPENDING', 'RETAIL SECTOR']},
    'XLP':  {'type':'keyword', 'themes':['EPU_ECONOMY'],
             'keywords':['CONSUMER STAPLES', 'CONSUMER GOODS', 'DEFENSIVE STOCKS']},
    # XLI: ECON_TRADE(미동작) → ECON_STOCKMARKET
    'XLI':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['INDUSTRIAL SECTOR', 'INDUSTRIALS', 'MANUFACTURING', 'DEFENSE CONTRACTOR']},
    # XLU: ENERGY 테마 유지 (XLE에서 동작 확인), 키워드 단순화
    'XLU':  {'type':'keyword', 'themes':['ENERGY'],
             'keywords':['UTILITIES SECTOR', 'ELECTRIC UTILITY', 'UTILITY COMPANY', 'ELECTRIC POWER']},
    # XLRE: ECON_REALESTATE(미동작) → ECON_STOCKMARKET
    'XLRE': {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['REAL ESTATE', 'REIT', 'COMMERCIAL PROPERTY', 'HOUSING MARKET']},
    # XLB: COMMODITY(미동작) → ECON_STOCKMARKET
    'XLB':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['MATERIALS SECTOR', 'BASIC MATERIALS', 'CHEMICAL COMPANY', 'MINING']},

    # ── 채권 ETF: ECON_INTEREST_RATES(미존재) → EPU_ECONOMY ────────
    # EPU_ECONOMY = Baker-Bloom-Davis 경제정책 불확실성 태그
    # 금리/국채/통화정책 기사에 빈번하게 등장하는 것으로 확인
    'TLT':  {'type':'keyword', 'themes':['EPU_ECONOMY'],
             'keywords':['TREASURY', 'FEDERAL RESERVE', 'LONG-TERM BOND', 'BOND YIELD', '20-YEAR']},
    'AGG':  {'type':'keyword', 'themes':['EPU_ECONOMY'],
             'keywords':['INVESTMENT GRADE', 'BOND MARKET', 'CREDIT SPREAD', 'AGGREGATE BOND']},
    'SHY':  {'type':'keyword', 'themes':['EPU_ECONOMY'],
             'keywords':['2-YEAR TREASURY', 'SHORT-TERM BOND', 'FEDERAL RESERVE', 'FED FUNDS']},
    'TIP':  {'type':'keyword', 'themes':['EPU_ECONOMY'],
             'keywords':['TIPS', 'INFLATION PROTECTED', 'REAL YIELD', 'CPI', 'INFLATION RATE']},

    # ── 대안 ETF: ECON_GOLD/COMMODITY(미존재) → ECON_STOCKMARKET ──
    # GLD: COMEX, WORLD GOLD COUNCIL 등 금 시장 특화 기관명 키워드 사용
    'GLD':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['GOLD PRICE', 'WORLD GOLD COUNCIL', 'COMEX GOLD',
                         'GOLD FUTURES', 'SPOT GOLD', 'GOLD BULL']},
    # DBC: 원자재 지수 관련 기관/키워드
    'DBC':  {'type':'keyword', 'themes':['ECON_STOCKMARKET'],
             'keywords':['COMMODITY INDEX', 'RAW MATERIALS', 'COMMODITY MARKET', 'COMMODITY ETF']},
}

print(f'TICKER_TO_GDELT v2 로드 완료: {len(TICKER_TO_GDELT_V2)}개 종목')

# ── v2 매핑으로 커버리지 재테스트 ─────────────────────────────────
# 전역 TICKER_TO_GDELT를 v2로 임시 교체 후 집계
_orig = dict(TICKER_TO_GDELT)
TICKER_TO_GDELT.clear()
TICKER_TO_GDELT.update(TICKER_TO_GDELT_V2)

print('\n집계 중 (v2)...')
result_v2 = aggregate_gdelt_by_stock(gkg_full)

# 원래 v1 복원
TICKER_TO_GDELT.clear()
TICKER_TO_GDELT.update(_orig)

cnt_v2 = result_v2.groupby('ticker')['gdelt_event_cnt'].sum().reindex(TICKERS, fill_value=0)
n_v2   = (cnt_v2 > 0).sum()
n_v1   = (cnt_full > 0).sum()
print(f'\n=== 커버리지 비교 ===')
print(f'  v1 (기존): {n_v1}/{len(TICKERS)}개 ({n_v1/len(TICKERS)*100:.0f}%)')
print(f'  v2 (개선): {n_v2}/{len(TICKERS)}개 ({n_v2/len(TICKERS)*100:.0f}%)')

print('\n=== 신규 커버 / 변화 종목 ===')
print(f'{"티커":6s}  {"v1건수":>6s}  {"v2건수":>6s}  변화')
for t in TICKERS:
    v1 = int(cnt_full.get(t, 0))
    v2 = int(cnt_v2.get(t, 0))
    if v1 != v2:
        arrow = '↑ 신규' if v1 == 0 and v2 > 0 else ('↑' if v2 > v1 else '↓')
        print(f'{t:6s}  {v1:6d}  {v2:6d}  {arrow}')

print('\n[미커버 종목 (v2 기준)]')
print([t for t in TICKERS if cnt_v2.get(t, 0) == 0])

## 다음 단계

### 완료 항목
- [x] `coverage=True` 1일치 수집 및 파이프라인 검증
- [x] AMZN `org_themed` 매핑 수정 (지리명 오매핑 차단)
- [x] 채권 ETF / GLD / 섹터 ETF 테마 코드 오류 원인 분석
- [x] TICKER_TO_GDELT v2 작성 및 커버리지 재검증

### 남은 작업

1. **v2 매핑 검증 완료 후 `ML_DataPrep_Example.ipynb` 반영**
   - `TICKER_TO_GDELT_V2`를 해당 노트북의 Step 5-C에 적용
   - 시뮬레이션 블록 → `aggregate_gdelt_by_stock()` 실 데이터 호출로 교체

2. **전체 기간 수집 스크립트 작성** (`GDELT_Collection.ipynb`)
   - 2016-01-01 ~ 2025-12-31 (약 9년, 영업일 ~2,250일)
   - 날짜별 반복 수집 + Parquet 저장 (메모리 효율)
   - 수집 실패 날짜 재시도 로직 포함

3. **AAPL 노이즈 필터링 정밀화 검토**
   - Step 6-B 결과에서 노이즈 비율이 50% 초과 시
   - `names`를 `['APPLE INC', 'APPLE COMPUTER']`로 유지하되
   - `org_themed` 타입으로 변경 + `ECON_STOCKMARKET` 테마 AND 필터 추가 고려

4. **JNJ/PG `&` 인코딩 확인**
   - Step 6-B의 raw 검색 결과가 0건이면 해당 날짜 기사 부재 → 다른 날짜로 재확인 필요
   - 0건이 아니면 실제 기관명 형태를 v2 매핑에 반영